In [1]:
import re
import json
from itertools import combinations

import pandas as pd

In [ ]:
GT_PATH = "Ground_truth_OBD_dependency.xlsx"
GT_SHEET_NAME = "Matrix"  # name of the sheet containing the matrix
PRED_PATH = "result_dependency.json"

# Ground truth matrix convention (per the sheet's own legend):
#   Row    US_i = the "requires" story  -> the DEPENDENT
#   Column US_j = the story whose Create must happen first -> the PREREQUISITE
#   Cell filled (1) at (row=US_i, col=US_j) means: US_i requires US_j
#     i.e. prerequisite = US_j (column), dependent = US_i (row)
DIRECTED_GT = True
ROW_IS_PREREQUISITE = False   # row is the DEPENDENT, not the prerequisite

ID_WIDTH = 3  # zero-padding width used in the JSON ids (US-002 -> width 3)

## Helper functions

In [1]:
def normalize_id(raw) -> str:
    """Turn 'US_1', 'US-1', 'US-001', 1, etc. into a canonical 'US-001' form."""
    s = str(raw).strip()
    m = re.search(r"(\d+)", s)
    if not m:
        raise ValueError(f"Could not extract a story number from id: {raw!r}")
    num = int(m.group(1))
    return f"US-{num:0{ID_WIDTH}d}"

In [2]:
def load_ground_truth(path: str):
    """Return (gt_pairs, gt_directed).
    gt_pairs: set of frozen sets (undirected pairs marked as dependent).
    gt_directed: set of (prereq, dependent) tuples (only if DIRECTED_GT)."""
    df = pd.read_excel(path, sheet_name=GT_SHEET_NAME, index_col=0)

    gt_pairs = set()
    gt_directed = set()

    for row_label in df.index:
        for col_label in df.columns:
            val = df.loc[row_label, col_label]
            if pd.isna(val):
                continue
            val_str = str(val).strip()
            if val_str in ("1", "1.0") or val is True:
                try:
                    i = normalize_id(row_label)
                    j = normalize_id(col_label)
                except ValueError:
                    continue
                if i == j:
                    continue
                gt_pairs.add(frozenset({i, j}))
                if DIRECTED_GT:
                    if ROW_IS_PREREQUISITE:
                        gt_directed.add((i, j))
                    else:
                        gt_directed.add((j, i))

    return gt_pairs, gt_directed

In [3]:
def load_predictions(path: str):
    """Return (pred_pairs, pred_directed) from the pipeline's result JSON."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    pred_pairs = set()
    pred_directed = set()

    for item in data:
        deps = item.get("final", {}).get("dependencies", [])
        for dep in deps:
            p = normalize_id(dep["prerequisite_id"])
            d = normalize_id(dep["dependent_id"])
            pred_pairs.add(frozenset({p, d}))
            pred_directed.add((p, d))

    return pred_pairs, pred_directed

In [4]:
def collect_all_ids(gt_pairs, pred_pairs):
    ids = set()
    for pair in gt_pairs | pred_pairs:
        ids |= set(pair)
    return sorted(ids, key=lambda x: int(re.search(r"\d+", x).group()))


def prf(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1


def evaluate(gt_pairs, pred_pairs, all_ids):
    all_pairs = {frozenset(c) for c in combinations(all_ids, 2)}

    tp_set = gt_pairs & pred_pairs
    fp_set = pred_pairs - gt_pairs
    fn_set = gt_pairs - pred_pairs
    tn_set = all_pairs - (gt_pairs | pred_pairs)

    tp, fp, fn, tn = len(tp_set), len(fp_set), len(fn_set), len(tn_set)
    precision, recall, f1 = prf(tp, fp, fn)
    accuracy = (tp + tn) / len(all_pairs) if all_pairs else 0.0

    return {
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy,
        "tp_set": tp_set, "fp_set": fp_set, "fn_set": fn_set,
    }


def evaluate_direction(gt_directed, pred_directed, tp_undirected_pairs):
    """Among pairs both sides agree are dependent, how often does direction match?"""
    if not gt_directed:
        return None
    correct = 0
    checked = 0
    mismatches = []
    for pair in tp_undirected_pairs:
        a, b = tuple(pair)
        gt_dir = (a, b) if (a, b) in gt_directed else (b, a) if (b, a) in gt_directed else None
        pred_dir = (a, b) if (a, b) in pred_directed else (b, a) if (b, a) in pred_directed else None
        if gt_dir is None or pred_dir is None:
            continue
        checked += 1
        if gt_dir == pred_dir:
            correct += 1
        else:
            mismatches.append((gt_dir, pred_dir))
    acc = correct / checked if checked else 0.0
    return {"checked": checked, "correct": correct, "accuracy": acc, "mismatches": mismatches}


def fmt_pair(pair):
    return " <-> ".join(sorted(pair))

## Load data

In [ ]:
gt_pairs, gt_directed = load_ground_truth(GT_PATH)
pred_pairs, pred_directed = load_predictions(PRED_PATH)
all_ids = collect_all_ids(gt_pairs, pred_pairs)

print(f"Ground truth dependency pairs : {len(gt_pairs)}")
print(f"Predicted dependency pairs    : {len(pred_pairs)}")
print(f"Stories involved               : {len(all_ids)}")

## Pair-level (undirected) evaluation

In [ ]:
results = evaluate(gt_pairs, pred_pairs, all_ids)

print(f"TP={results['tp']}  FP={results['fp']}  FN={results['fn']}  TN={results['tn']}")
print(f"Precision: {results['precision']:.3f}")
print(f"Recall   : {results['recall']:.3f}")
print(f"F1       : {results['f1']:.3f}")
print(f"Accuracy : {results['accuracy']:.3f}")

In [ ]:
summary_df = pd.DataFrame([{
    "TP": results["tp"], "FP": results["fp"], "FN": results["fn"], "TN": results["tn"],
    "Precision": round(results["precision"], 3),
    "Recall": round(results["recall"], 3),
    "F1": round(results["f1"], 3),
    "Accuracy": round(results["accuracy"], 3),
}])
summary_df

### False positives (predicted but not in ground truth)

In [ ]:
fp_df = pd.DataFrame(sorted(fmt_pair(p) for p in results["fp_set"]), columns=["pair"])
fp_df

### False negatives (in ground truth but missed)

In [ ]:
fn_df = pd.DataFrame(sorted(fmt_pair(p) for p in results["fn_set"]), columns=["pair"])
fn_df

## Direction evaluation

Among the pairs both ground truth and prediction agree are dependent (the TP
set above), how often does the pipeline get the *direction* right too?
This is the stricter, more meaningful metric since a dependency's direction
is the whole point of a prerequisite relationship.

In [ ]:
if DIRECTED_GT:
    dir_results = evaluate_direction(gt_directed, pred_directed, results["tp_set"])
    if dir_results:
        print(f"Checked: {dir_results['checked']}  Correct: {dir_results['correct']}")
        print(f"Direction accuracy: {dir_results['accuracy']:.3f}")
        if dir_results["mismatches"]:
            print("\nMismatches (ground_truth_dir vs predicted_dir):")
            for gt_dir, pred_dir in dir_results["mismatches"]:
                print(f"  - GT: {gt_dir[0]} -> {gt_dir[1]} | Pred: {pred_dir[0]} -> {pred_dir[1]}")
else:
    print("DIRECTED_GT is False - skipping direction evaluation.")

## Confusion matrix, shown as a grid (same layout as the ground truth sheet)

Reconstructs the full US_i \ US_j grid and colors each cell by outcome:

- **TP** (green) — dependency correctly found, direction correct
- **TP (dir ✗)** (amber) — pair correctly found to be dependent, but the predicted direction was wrong
- **FP** (red) — pipeline predicted a dependency that isn't in the ground truth
- **FN** (yellow) — ground truth has a dependency the pipeline missed
- **–** (grey) — diagonal / not applicable
- blank — true negative (both agree there's no dependency); left blank since there are usually many of these

Cell position follows the ground truth's own convention: **row = dependent ("requires"), column = prerequisite**.

In [ ]:
def find_direction(pair, directed_set):
    """Given an undirected pair (frozenset) and a set of (prereq, dependent)
    tuples, return the (prereq, dependent) tuple if found, else None."""
    a, b = tuple(pair)
    if (a, b) in directed_set:
        return (a, b)
    if (b, a) in directed_set:
        return (b, a)
    return None


def pair_cell_position(pair, gt_directed, pred_directed):
    """Decide which grid cell (row, col) represents this pair, matching the
    ground truth's row=dependent / col=prerequisite convention. Falls back to
    numeric ordering (larger id = row) if no directed info is available."""
    direction = find_direction(pair, gt_directed) or find_direction(pair, pred_directed)
    if direction:
        prereq, dependent = direction
        return dependent, prereq  # row, col
    a, b = sorted(pair, key=lambda x: int(re.search(r"\d+", x).group()))
    return b, a  # larger-numbered id as row, matching the sheet's triangular pattern

In [ ]:
# Re-read the ground truth just to get the full, original ID ordering (US_1..US_N)
gt_raw = pd.read_excel(GT_PATH, sheet_name=GT_SHEET_NAME, index_col=0)
full_ids = [normalize_id(x) for x in gt_raw.index]

matrix_df = pd.DataFrame("", index=full_ids, columns=full_ids)
for i in full_ids:
    matrix_df.loc[i, i] = "-"

for pair in results["tp_set"]:
    row, col = pair_cell_position(pair, gt_directed, pred_directed)
    gt_dir = find_direction(pair, gt_directed)
    pred_dir = find_direction(pair, pred_directed)
    label = "TP"
    if DIRECTED_GT and gt_dir and pred_dir and gt_dir != pred_dir:
        label = "TP (dir X)"
    matrix_df.loc[row, col] = label

for pair in results["fp_set"]:
    row, col = pair_cell_position(pair, gt_directed, pred_directed)
    matrix_df.loc[row, col] = "FP"

for pair in results["fn_set"]:
    row, col = pair_cell_position(pair, gt_directed, pred_directed)
    matrix_df.loc[row, col] = "FN"

matrix_df

In [ ]:
COLOR_MAP = {
    "TP": "background-color: #6aa84f; color: white",
    "TP (dir X)": "background-color: #e69138; color: white",
    "FP": "background-color: #cc4125; color: white",
    "FN": "background-color: #f1c232; color: black",
    "-": "background-color: #d9d9d9; color: #666666",
}

def highlight(val):
    return COLOR_MAP.get(val, "")

try:
    styled_matrix = matrix_df.style.map(highlight)  # pandas >= 2.1
except AttributeError:
    styled_matrix = matrix_df.style.applymap(highlight)  # older pandas
styled_matrix = styled_matrix.set_properties(**{"text-align": "center"})
styled_matrix

### Export the colored matrix to Excel

In [ ]:
styled_matrix.to_excel("evaluation_matrix.xlsx", engine="openpyxl")
print("Saved: evaluation_matrix.xlsx")